In [4]:
import os

from dotenv import load_dotenv

# Document Loader
from langchain_community.document_loaders import WebBaseLoader

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector Store
from langchain_community.vectorstores import FAISS

# LLM
from langchain_groq import ChatGroq

In [5]:
load_dotenv()

os.environ["USER_AGENT"] = "News-RAG-Chatbot"

groq_api = os.getenv("GROQ_API_KEY")

print("Groq API Loaded Successfully ✅")

Groq API Loaded Successfully ✅


# 📰 Step 3: Load BBC News Website

In this step, we load the BBC News homepage using LangChain's WebBaseLoader and inspect the extracted content before preprocessing.

In [6]:
url = "https://www.bbc.com/news"

loader = WebBaseLoader(url)

documents = loader.load()

print(f"Total Documents : {len(documents)}")

Total Documents : 1


In [7]:
print(documents[0].metadata)

{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world', 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.', 'language': 'en-GB'}


In [8]:
print(documents[0].page_content[:1500])

BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyHomeNewsUS & CanadaUKUK PoliticsEnglandN. IrelandN. Ireland PoliticsScotlandScotland PoliticsWalesWales PoliticsAfricaAsiaChinaIndiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyFootball 2026SportBusinessWorld of BusinessTechnology of BusinessNYSE Opening BellTechnologyArtificial IntelligenceIntelligence RevolutionAI v the MindTech NowHealthCultureFilm & TVMusicArt & DesignStyleBooksEntertainment NewsArtsArts in MotionTravelDestinationsAfricaAntarcticaAsiaAustralia and PacificCaribbean & BermudaCentral AmericaEuropeMiddle EastNorth AmericaSouth AmericaWorld’s TableCulture & ExperiencesAdventuresThe SpeciaListEarthScienceNatural WondersClimate Solu

In [9]:
documents[0].metadata

{'source': 'https://www.bbc.com/news',
 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world',
 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.',
 'language': 'en-GB'}

In [10]:
print(documents[0].page_content[:2000])

BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyHomeNewsUS & CanadaUKUK PoliticsEnglandN. IrelandN. Ireland PoliticsScotlandScotland PoliticsWalesWales PoliticsAfricaAsiaChinaIndiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyFootball 2026SportBusinessWorld of BusinessTechnology of BusinessNYSE Opening BellTechnologyArtificial IntelligenceIntelligence RevolutionAI v the MindTech NowHealthCultureFilm & TVMusicArt & DesignStyleBooksEntertainment NewsArtsArts in MotionTravelDestinationsAfricaAntarcticaAsiaAustralia and PacificCaribbean & BermudaCentral AmericaEuropeMiddle EastNorth AmericaSouth AmericaWorld’s TableCulture & ExperiencesAdventuresThe SpeciaListEarthScienceNatural WondersClimate Solu

In [11]:
content = documents[0].page_content

print("Characters :", len(content))
print("Words :", len(content.split()))

Characters : 10814
Words : 1493


In [12]:
type(documents[0])

langchain_core.documents.base.Document

In [13]:
dir(documents[0])

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_extra_info__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__

# ✂️ Step 4: Document Chunking

Large Language Models have context window limitations. Instead of embedding the entire webpage as a single document, we split it into smaller overlapping chunks.

Benefits:
- Better semantic search
- Higher retrieval accuracy
- Reduced token usage
- Improved answer quality

In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

In [15]:
chunks = splitter.split_documents(documents)

print(f"Total Chunks Created : {len(chunks)}")

Total Chunks Created : 14


In [16]:
print(chunks[0].page_content)

BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyHomeNewsUS & CanadaUKUK PoliticsEnglandN. IrelandN. Ireland PoliticsScotlandScotland PoliticsWalesWales PoliticsAfricaAsiaChinaIndiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyFootball 2026SportBusinessWorld of BusinessTechnology of BusinessNYSE Opening BellTechnologyArtificial IntelligenceIntelligence RevolutionAI v the MindTech NowHealthCultureFilm & TVMusicArt & DesignStyleBooksEntertainment NewsArtsArts in MotionTravelDestinationsAfricaAntarcticaAsiaAustralia and PacificCaribbean & BermudaCentral AmericaEuropeMiddle EastNorth AmericaSouth AmericaWorld’s TableCulture & ExperiencesAdventuresThe SpeciaListEarthScienceNatural WondersClimate


In [17]:
chunks[0].metadata

{'source': 'https://www.bbc.com/news',
 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world',
 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.',
 'language': 'en-GB'}

In [18]:
for i in range(3):
    print(f"\nChunk {i+1}")
    print("-" * 50)
    print(f"Characters : {len(chunks[i].page_content)}")
    print(chunks[i].page_content[:300])


Chunk 1
--------------------------------------------------
Characters : 995
BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesB

Chunk 2
--------------------------------------------------
Characters : 999
and PacificCaribbean & BermudaCentral AmericaEuropeMiddle EastNorth AmericaSouth AmericaWorld’s TableCulture & ExperiencesAdventuresThe SpeciaListEarthScienceNatural WondersClimate SolutionsSustainable BusinessGreen LivingAudioPodcast CategoriesRadioAudio FAQsVideoBBC MaestroDiscover the WorldLiveLi

Chunk 3
--------------------------------------------------
Characters : 998
Iran and Iraq over the coming days.19 hrs agoMiddle EastAustralia probes mystery space balls that washed up on beachOfficials are searching for the origins of six piec

In [19]:
type(chunks[0])

langchain_core.documents.base.Document

In [20]:
print(f"Original Documents : {len(documents)}")
print(f"Chunks Created     : {len(chunks)}")

Original Documents : 1
Chunks Created     : 14
